In [1]:
import os
import numpy as np
import pandas as pd
from scipy import interpolate
from scipy.signal import resample
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [4]:
# root dir
root = 'TDBRAIN\derivatives'
# participants file path
ptc_path = os.path.join(root, 'TDBRAIN_participants_V2.tsv')
ptc = pd.read_csv(ptc_path, sep='\t')
ptc

,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,BDI_post,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110
0,sub-19681349,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
1,sub-19681385,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
2,sub-19684666,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
3,sub-19686324,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
4,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,3,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1346,sub-88079017,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1347,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Labels

In [5]:
# delete invalid participants and the participants with NAN participants_ID
ptc = ptc[~(ptc['participants_ID'].isna())]
ptc

,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,BDI_post,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110
0,sub-19681349,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
1,sub-19681385,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
2,sub-19684666,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
3,sub-19686324,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
4,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,3,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1342,sub-88078657,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1343,sub-88078749,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1344,sub-88078793,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1345,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Get duplicate values of participants_id
l = ptc.drop_duplicates(['participants_ID'],keep=False)['participants_ID'].to_list()
dup_ptc = ptc[~(ptc['participants_ID'].isin(l))]
dup_ptc

,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,BDI_post,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110
4,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,3,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
5,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,3,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
7,sub-19690494,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
8,sub-19690494,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
9,sub-19690969,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
958,sub-88047289,DISCOVERY,DEPERSONALIZATION,UNKNOWN,NaN,YES,summer,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
962,sub-88047649,DISCOVERY,OCD,OCD,OCD,YES,summer,NaN,1,1,...,6,7,NaN,NaN,NaN,NaN,NaN,23,12,NaN
963,sub-88047649,DISCOVERY,OCD,OCD,OCD,YES,summer,NaN,1,1,...,6,7,NaN,NaN,NaN,NaN,NaN,23,12,NaN
972,sub-88048193,DISCOVERY,OCD,OCD,OCD,YES,summer,NaN,1,0,...,0,7,NaN,NaN,NaN,NaN,NaN,22,13,NaN


In [7]:
# Test if the same participant has different indications for different sessions
for i in range(len(dup_ptc)):
    temp = dup_ptc[dup_ptc['participants_ID']==dup_ptc['participants_ID'].iloc[i]].drop_duplicates(['indication'])
    if len(temp)!=1:
        print(len(temp))
        print('The different sessions in the same participant may have different indications!')

2
The different sessions in the same participant may have different indications!
2
The different sessions in the same participant may have different indications!


In [8]:
# EEG data preprocessing

# all disease mapping
all_dise_dict = {}
ls_ind = list(ptc['indication'].unique())
for i in range(len(ls_ind)):
    all_dise_dict[ls_ind[i]] = i
print(all_dise_dict)
    
ptc['indication_id'] = ptc['indication'].map(all_dise_dict)
ptc

{'REPLICATION': 0, 'BURNOUT': 1, 'SMC': 2, 'HEALTHY': 3, 'Dyslexia': 4, 'CHRONIC PAIN': 5, 'MDD': 6, nan: 7, 'ADHD': 8, 'ADHD/ASPERGER': 9, 'PDD NOS/DYSLEXIA': 10, 'PDD NOS': 11, 'WHIPLASH': 12, 'ANXIETY': 13, 'ADHD/DYSLEXIA': 14, 'ASD': 15, 'TINNITUS': 16, 'OCD': 17, 'Tinnitus': 18, 'PDD NOS ': 19, 'PANIC': 20, 'MDD/ANXIETY': 21, 'MIGRAINE': 22, 'PDD NOS/ANXIETY': 23, 'PARKINSON': 24, 'BIPOLAR': 25, 'MDD/bipolar': 26, 'DYSPRAXIA': 27, 'TINNITUS/MDD': 28, 'ADHD/ASD/ANXIETY': 29, 'MDD/ADHD': 30, 'ADHD/PDD NOS': 31, 'MDD/BIPOLAR': 32, 'ASPERGER': 33, 'ADHD/EPILEPSY': 34, 'MDD/PAIN': 35, 'PDD NOS/GTS': 36, 'PDD NOS/ADHD': 37, 'PDD NOS/ASD': 38, 'TBI': 39, 'ADHD/ANXIETY': 40, 'ADHD/DYSLEXIA/DYSCALCULIA': 41, 'ADHD/MDD': 42, 'MDD/PANIC': 43, 'DEPERSONALIZATION': 44, 'MDD/TRAUMA': 45, 'PTSD/ADHD': 46, 'OCD/DPS': 47, 'MDD/OCD': 48, 'MDD/TUMOR': 49, 'ADHD/GTS': 50, 'OCD/MDD': 51, 'CONVERSION DX': 52, 'ASD/ASPERGER': 53, 'MDD/ADHD/LYME': 54, 'ADHD/OCD': 55, 'MSA-C': 56, 'OCD/ASD': 57, 'STROKE/P

,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110,indication_id
0,sub-19681349,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
1,sub-19681385,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
2,sub-19684666,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
3,sub-19686324,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
4,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,3,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1342,sub-88078657,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1343,sub-88078749,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1344,sub-88078793,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1345,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63


In [9]:
# EEG data preprocessing

# all disease mapping
all_dise_dict = {}
ls_ind = list(ptc['indication'].unique())
for i in range(len(ls_ind)):
    all_dise_dict[ls_ind[i]] = i
print(all_dise_dict)
    
ptc['indication_id'] = ptc['indication'].map(all_dise_dict)
ptc

{'REPLICATION': 0, 'BURNOUT': 1, 'SMC': 2, 'HEALTHY': 3, 'Dyslexia': 4, 'CHRONIC PAIN': 5, 'MDD': 6, nan: 7, 'ADHD': 8, 'ADHD/ASPERGER': 9, 'PDD NOS/DYSLEXIA': 10, 'PDD NOS': 11, 'WHIPLASH': 12, 'ANXIETY': 13, 'ADHD/DYSLEXIA': 14, 'ASD': 15, 'TINNITUS': 16, 'OCD': 17, 'Tinnitus': 18, 'PDD NOS ': 19, 'PANIC': 20, 'MDD/ANXIETY': 21, 'MIGRAINE': 22, 'PDD NOS/ANXIETY': 23, 'PARKINSON': 24, 'BIPOLAR': 25, 'MDD/bipolar': 26, 'DYSPRAXIA': 27, 'TINNITUS/MDD': 28, 'ADHD/ASD/ANXIETY': 29, 'MDD/ADHD': 30, 'ADHD/PDD NOS': 31, 'MDD/BIPOLAR': 32, 'ASPERGER': 33, 'ADHD/EPILEPSY': 34, 'MDD/PAIN': 35, 'PDD NOS/GTS': 36, 'PDD NOS/ADHD': 37, 'PDD NOS/ASD': 38, 'TBI': 39, 'ADHD/ANXIETY': 40, 'ADHD/DYSLEXIA/DYSCALCULIA': 41, 'ADHD/MDD': 42, 'MDD/PANIC': 43, 'DEPERSONALIZATION': 44, 'MDD/TRAUMA': 45, 'PTSD/ADHD': 46, 'OCD/DPS': 47, 'MDD/OCD': 48, 'MDD/TUMOR': 49, 'ADHD/GTS': 50, 'OCD/MDD': 51, 'CONVERSION DX': 52, 'ASD/ASPERGER': 53, 'MDD/ADHD/LYME': 54, 'ADHD/OCD': 55, 'MSA-C': 56, 'OCD/ASD': 57, 'STROKE/P

,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110,indication_id
0,sub-19681349,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
1,sub-19681385,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
2,sub-19684666,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
3,sub-19686324,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
4,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,3,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1342,sub-88078657,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1343,sub-88078749,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1344,sub-88078793,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1345,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63


In [10]:
# Drop duplicate participants
ptc = ptc.drop_duplicates(['participants_ID'])
ptc

,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110,indication_id
0,sub-19681349,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
1,sub-19681385,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
2,sub-19684666,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,1,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
3,sub-19686324,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,2,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
4,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,3,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1342,sub-88078657,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1343,sub-88078749,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1344,sub-88078793,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63
1345,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63


In [11]:
# label.npy
# initialization column 'subject_id'
ptc['subject_id'] = np.arange(1, ptc.shape[0] + 1)
ptc

,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110,indication_id,subject_id
0,sub-19681349,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0,1
1,sub-19681385,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0,2
2,sub-19684666,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0,3
3,sub-19686324,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0,4
4,sub-19687321,REPLICATION,REPLICATION,REPLICATION,REPLICATION,YES,REPLICATION,REPLICATION,REPLICATION,REPLICATION,...,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,REPLICATION,NaN,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1342,sub-88078657,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63,1269
1343,sub-88078749,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63,1270
1344,sub-88078793,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63,1271
1345,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63,1272


In [12]:
# build subject info: "sub-XXX" -> (label, pid)
sub_info = {}  # key: subject name, value: (label, pid)
for row in ptc.itertuples(index=False):
    sub_name = row[-1]          
    pid = int(sub_name)   # convert '001' -> 1
    label = row[-2]   
    sub_info[sub_name] = (label, pid)

len(sub_info), list(sub_info.items())[:5]

(1273, [(1, (0, 1)), (2, (0, 2)), (3, (0, 3)), (4, (0, 4)), (5, (0, 5))])

## Features

In [13]:
def data_preprocessing(
    eeg_data: np.ndarray,
    ch_names: list,
    source_sample_rate: int,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 45.0,
    verbose=True
):
    """
    Clean EEG data using bandpass filtering, percentile-based bad channel detection,
    ICA + ICLabel artifact removal, resampling, re-referencing, epoching, and z-score normalization.

    Args:
        eeg_data (np.ndarray): EEG data, shape (T, C).
        ch_names (list): List of channel names.
        source_sample_rate (int): Original sampling frequency of the EEG data.
        notch_freq (float): Notch filter frequency (default: 60 Hz).
        l_freq (float): Low cutoff frequency for bandpass filter (default: 0.5 Hz).
        h_freq (float): High cutoff frequency for bandpass filter (default: 40 Hz).
        verbose (bool): Verbose output.

    Returns:
        mne.io.Raw: Preprocessed MNE Raw object.
    """
    # 1. Construct MNE Raw object
    info = mne.create_info(ch_names=ch_names, sfreq=source_sample_rate, ch_types=['eeg'] * len(ch_names))
    raw = mne.io.RawArray(eeg_data.T, info)

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))
    if verbose:
        print("✔ Montage set: 'standard_1020'.")
        
    # 3. Notch Filter at 50 Hz
    raw.notch_filter(freqs=notch_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Notch @ {notch_freq} Hz")

    #  Bandpass Filter (0.5–45 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    if verbose:
        print(f"✔ Bandpass filter applied ({l_freq}–{h_freq} Hz).")
        
    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=None, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [14]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [15]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
standard_channels = ['Fp1','Fp2','F7','F3','Fz','F4','F8','FC3','FCz','FC4','T7',
                     'C3','Cz','C4','T8','CP3','CPz','CP4','P7','P3','Pz','P4','P8','O1','Oz','O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")

sub_id = 1
ORIGINAL_SAMPLE_RATE = 500  # original sampling rate
ls_ptc = ptc['participants_ID'].to_list()
for sub in os.listdir(root):
    # select valid subjects
    if sub in ls_ptc:
        print(sub)
        print("Effective subject number: ", sub_id)
        sub_path = os.path.join(root, sub) 
        for ses in os.listdir(sub_path):
            print(ses)
            print('-----------')
            ses_path = os.path.join(sub_path, ses, 'eeg')
            for task in os.listdir(ses_path):
                if 'EC' in task:
                    print("Eye closed")
                else:
                    print("Eye open")
                task_path = os.path.join(ses_path, task)
                data = pd.read_csv(task_path, usecols=standard_channels).to_numpy()
                print("Raw data shape ", data.shape)
                T_raw = data.shape[0]
                for fs in SAMPLE_RATE_LIST:
                    T_res = int(T_raw * fs / ORIGINAL_SAMPLE_RATE)
                    # compute number of segments
                    n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                    subject_segment_counts[sub_id][fs] += n_seg
                    N_total += n_seg
                    print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                print('------------End of Task-------------------')
        print("Total segments for Session", sub_id, ":", sum(subject_segment_counts[sub_id][fs] for fs in SAMPLE_RATE_LIST))
        sub_id += 1
        print('--------------------------------------------------------------------------\n')
    else:
        print(" ", sub, " is not in the list of valid subjects.")
        print('--------------------------------------------------------------------------\n')
        continue
        
        
print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

sub-19681349
Effective subject number:  1
ses-1
-----------
Eye closed
Raw data shape  (60003, 26)
fs=200Hz: T_res=24001, STEP=200, n_seg=119
fs=100Hz: T_res=12000, STEP=200, n_seg=59
fs=50Hz: T_res=6000, STEP=200, n_seg=29
------------End of Task-------------------
Eye open
Raw data shape  (60014, 26)
fs=200Hz: T_res=24005, STEP=200, n_seg=119
fs=100Hz: T_res=12002, STEP=200, n_seg=59
fs=50Hz: T_res=6001, STEP=200, n_seg=29
------------End of Task-------------------
Total segments for Session 1 : 414
--------------------------------------------------------------------------

sub-19681385
Effective subject number:  2
ses-1
-----------
Eye closed
Raw data shape  (60015, 26)
fs=200Hz: T_res=24006, STEP=200, n_seg=119
fs=100Hz: T_res=12003, STEP=200, n_seg=59
fs=50Hz: T_res=6001, STEP=200, n_seg=29
------------End of Task-------------------
Eye open
Raw data shape  (60020, 26)
fs=200Hz: T_res=24008, STEP=200, n_seg=119
fs=100Hz: T_res=12004, STEP=2

In [16]:
output_root = os.path.join('Processed', sub_folder_path, 'TDBrain')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))
print("Created memmap for X with shape:", X_mm.shape)

X path: Processed\L400\TDBrain\X.dat
y path: Processed\L400\TDBrain\y.dat
Created memmap for X with shape: (555455, 400, 26)


## PASS 2: Process and store segments into memmap

In [17]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation

sub_id = 1
ORIGINAL_SAMPLE_RATE = 500  # original sampling rate
ls_ptc = ptc['participants_ID'].to_list()
for sub in os.listdir(root):
    # select valid subjects
    if sub in ls_ptc:
        print(sub)
        print("Effective subject number: ", sub_id)
        sub_label, sub_pid = sub_info[sub_id]
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            print(ses)
            print('-----------')
            ses_path = os.path.join(sub_path, ses, 'eeg')
            for task in os.listdir(ses_path):
                if 'EC' in task:
                    print("Eye closed")
                else:
                    print("Eye open")
                task_path = os.path.join(ses_path, task)
                data = pd.read_csv(task_path, usecols=standard_channels).to_numpy()
                print("Raw data shape ", data.shape)
                raw = data_preprocessing(
                    eeg_data=data,
                    ch_names=standard_channels,
                    source_sample_rate=ORIGINAL_SAMPLE_RATE,  # original sampling rate
                    notch_freq=50,
                    l_freq=0.5,
                    h_freq=45.0,
                    verbose=True
                )
                data = raw.get_data().T
                T_raw = data.shape[0]
                total_seconds_all += data.shape[0] / ORIGINAL_SAMPLE_RATE
                for fs in SAMPLE_RATE_LIST:
                    # resample to target fs
                    data_resampled = resample_time_series(data, ORIGINAL_SAMPLE_RATE, fs)
                    T_res, _ = data_resampled.shape
                    print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
        
                    # overlapped sliding window with fixed STEP (in timestamps)
                    starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                    print(f"fs={fs}Hz: segments={len(starts)}")
        
                    for s in starts:
                        if cur >= N_total:
                            raise RuntimeError("Exceeded predicted N_total.")
        
                        X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                        y_mm[cur, 0] = float(sub_label)       # label
                        y_mm[cur, 1] = float(sub_id)      # Global trial ID
                        y_mm[cur, 2] = float(fs)          # sample rate (scale)
                        cur += 1
                print('------------End of Task-------------------')
        print("Total segments for Session", sub_id, ":", sum(subject_segment_counts[sub_id][fs] for fs in SAMPLE_RATE_LIST))
        sub_id += 1
        print('--------------------------------------------------------------------------\n')
    else:
        print(" ", sub, " is not in the list of valid subjects.")
        print('--------------------------------------------------------------------------\n')
        continue
        
        
total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

sub-19681349
Effective subject number:  1
ses-1
-----------
Eye closed
Raw data shape  (60003, 26)
Creating RawArray with float64 data, n_channels=26, n_times=60003
    Range : 0 ... 60002 =      0.000 ...   120.004 secs
Ready.
✔ Montage set: 'standard_1020'.
✔ Notch @ 50 Hz
✔ Bandpass filter applied (0.5–45.0 Hz).
Fitting ICA to data using 26 channels (please be patient, this may take a while)
Selecting by non-zero PCA components: 26 components
Fitting ICA took 9.8s.
✔ ICA fitted.
Applying ICA to Raw instance
    Transforming to ICA space (26 components)
    Zeroing out 1 ICA component
    Projecting back using 26 PCA components
✔ ICA applied. Excluded components: [14]
fs=200Hz: resampled shape=(24001, 26)
fs=200Hz: segments=119
fs=100Hz: resampled shape=(12000, 26)
fs=100Hz: segments=59
fs=50Hz: resampled shape=(6000, 26)
fs=50Hz: segments=29
------------End of Task-------------------
Eye open
Raw data shape  (60014, 26)
Creating RawArray with float64 data, n_channels=26, n_times=600

## Create JSON file

In [18]:
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "MONTAGE": "standard_1020",
    "CHANNELS": standard_channels,
    "LABELS": {
        "disease_id": {0: "Healthy Control (HC)"},  # too much others, ignore
        "subject_id": "integer from 1 to number of subjects",
        "sampling_frequency": "integer in SAMPLE_RATE_LIST"
    }
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved meta: Processed\L400\TDBrain\meta.json


## Load and check the processed data

In [19]:
# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

total_samples = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject
    total_samples += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total samples:", total_samples)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 555455
T: 400
C: 26
SAMPLE_RATE_LIST: [200, 100, 50]
MONTAGE: standard_1020
CHANNELS: ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC3', 'FCz', 'FC4', 'T7', 'C3', 'Cz', 'C4', 'T8', 'CP3', 'CPz', 'CP4', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2']
LABELS: {'disease_id': {'0': 'Healthy Control (HC)'}, 'subject_id': 'integer from 1 to number of subjects', 'sampling_frequency': 'integer in SAMPLE_RATE_LIST'}
-------------------------------------
Subject ID 001: X shape=(414, 400, 26), y shape=(414, 3)
Subject ID 002: X shape=(414, 400, 26), y shape=(414, 3)
Subject ID 003: X shape=(414, 400, 26), y shape=(414, 3)
Subject ID 004: X shape=(414, 400, 26), y shape=(414, 3)
Subject ID 005: X shape=(828, 400, 26), y shape=(828, 3)
Subject ID 006: X shape=(414, 400, 26), y shape=(414, 3)
Subject ID 007: X shape=(828, 400, 26), y shape=(828, 3)
Subject ID 008: X shape=(825, 400, 26), y shape=(825, 3)
Subject ID 009: X shape=(828, 400, 26), y shape=(828, 3)
Subject ID 010: X shape=(411